# Замер Random Forest: последние 75% train

В Kaggle включите GPU T4 x2, подключите train_solo_track.parquet и test_solo_track.parquet и выполните Save & Run All.
Нужны установленные CuPy, cuML, pandas, PyArrow и sklearn. Ноутбук автономный, другие файлы проекта не нужны.

По умолчанию: h1, direct и residual, параметры из stacking_part_1; без подбора.
Train содержит только метки до 18 октября 2025. Первые floor(N / 4) обучающих дат исключены,
на оставшихся датах используются все получасовые старты. Более ранние факты остаются доступными для лагов.
Проверка: 18–31 октября, только 10:30. Это замер времени, не основание для выбора модели по test.
Можно задать BENCHMARK_HORIZONS=(1,8), чтобы дополнительно замерить h8, либо BOUNDARY='2025-10-11'
и QUERY_DAYS=7 для точного воспроизведения последней OOF-недели.

Отдельно измеряются загрузка, общие признаки, признаки горизонта, корреляционные лаги,
сборка и обработка матриц, fit, predict train и predict test. GPU синхронизируется на границах замеров.
Результаты каждого законченного этапа сразу записываются в /kaggle/working/rf_benchmark.
Это диагностический ноутбук: он не продолжает прерванный fit и не создаёт результаты для сборщика stacking.

In [ ]:
import time
from pathlib import Path
BENCHMARK_START = time.monotonic()
BENCHMARK_HORIZONS = (1,)
VARIANTS = ('direct', 'residual')
BOUNDARY = '2025-10-18'
QUERY_DAYS = 14
INPUT_DIR = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working/rf_benchmark'
PROFILE = 'recent75'
MODE = 'oof'
ROLE = 'worker'
PARAMS = {'rf_direct': {'1': {'algorithm': 'random_forest', 'fold_scores': [0.26783833607079893, 0.35015051786120815, 0.27551559034755535], 'horizon': 1, 'params': {'max_depth': 10, 'max_features': 0.6, 'max_samples': 0.8, 'min_samples_leaf': 32, 'n_estimators': 150}, 'residual': False, 'selection_end_exclusive': '2025-09-20 00:00:00', 'selection_labels_through': '2025-09-19 11:00:00', 'trees': 150, 'trial': 0, 'tuning_score': 0.2978348147598541}, '8': {'algorithm': 'random_forest', 'fold_scores': [0.37831892703498315, 0.3710947695256506, 0.3499387871323048], 'horizon': 8, 'params': {'max_depth': 12, 'max_features': 0.7252813963310067, 'max_samples': 0.5852620618436457, 'min_samples_leaf': 18, 'n_estimators': 300}, 'residual': False, 'selection_end_exclusive': '2025-09-20 00:00:00', 'selection_labels_through': '2025-09-19 14:30:00', 'trees': 300, 'trial': 4, 'tuning_score': 0.36645082789764616}}, 'rf_residual': {'1': {'algorithm': 'random_forest', 'fold_scores': [0.32227336937702133, 0.41038809352071964, 0.28576469765737444], 'horizon': 1, 'params': {'max_depth': 12, 'max_features': 0.7102471755674296, 'max_samples': 0.5102922471479012, 'min_samples_leaf': 84, 'n_estimators': 222}, 'residual': True, 'selection_end_exclusive': '2025-09-20 00:00:00', 'selection_labels_through': '2025-09-19 11:00:00', 'trees': 222, 'trial': 0, 'tuning_score': 0.33947538685170514}, '8': {'algorithm': 'random_forest', 'fold_scores': [0.39148559333574595, 0.38481879720446893, 0.336581951348399], 'horizon': 8, 'params': {'max_depth': 10, 'max_features': 0.3608406803967395, 'max_samples': 0.5325257964926398, 'min_samples_leaf': 85, 'n_estimators': 500}, 'residual': True, 'selection_end_exclusive': '2025-09-20 00:00:00', 'selection_labels_through': '2025-09-19 14:30:00', 'trees': 500, 'trial': 2, 'tuning_score': 0.3709621139628713}}}

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

START = pd.Timestamp('2025-09-20')
TEST_START = pd.Timestamp('2025-10-18')
DAY = pd.Timedelta(days=1)
STEP = pd.Timedelta(minutes=30)
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
WINDOW_OFFSETS = tuple(range(-4, 5))
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
TRAIN_MINUTES = (570, 600, 630, 660, 690)
REGIMES = {'five_starts':('five_starts',), 'recent75':('recent_all',)}[PROFILE]
STACK_WEEKS = (2, 3, 4)
VALIDATION_WEEKS = STACK_WEEKS[1:]
FAMILIES = ('rf', 'light_gbm', 'catboost', 'xgb', 'ridge')
MODEL_NAMES = [f'{f}_{v}' for f in FAMILIES for v in ('direct', 'residual')]
KEY = ['cutoff', 'route_id', 'horizon']

TARGET_SCALE = 100000
ETA_CONTEXT = None
THREADS = min(4, os.cpu_count() or 1)

def file_digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def locate(name):
    hits = list(INPUT.rglob(name))
    if len(hits) != 1:
        raise ValueError(f'Ожидается один {name}, найдено {hits}')
    return hits[0]

def initialize():
    global INPUT, OUTPUT, ROOT, TRAIN_PATH, TEST_PATH, RUN_ID, CHECKPOINTS
    INPUT = Path(INPUT_DIR)
    TRAIN_PATH = locate('train_solo_track.parquet')
    TEST_PATH = locate('test_solo_track.parquet')
    fingerprints = [file_digest(TRAIN_PATH),file_digest(TEST_PATH)]
    identity = [CODE_ID,PROFILE,str(ROUTE_ALPHA),*fingerprints]
    if not np.isfinite(ROUTE_ALPHA) or ROUTE_ALPHA<=0:
        raise ValueError('ROUTE_ALPHA должен быть положительным')
    RUN_ID = hashlib.sha256('|'.join(identity).encode()).hexdigest()[:24]
    OUTPUT = Path(OUTPUT_DIR)
    ROOT = OUTPUT / f'stack_{RUN_ID}'
    ROOT.mkdir(parents=True, exist_ok=True)
    CHECKPOINTS = [p for p in INPUT.rglob(ROOT.name) if p.is_dir() and p.resolve() != ROOT.resolve()]
    print(f'Конфигурация {RUN_ID}, часть {PART}, режим {MODE}, роль {ROLE}')

def profile_path(rel):
    return str(rel)

def existing(rel):
    rel = profile_path(rel)
    dest = ROOT / rel
    if dest.is_file():
        return dest
    hits = [base / rel for base in CHECKPOINTS if (base / rel).is_file()]
    if not hits:
        return None
    if len(hits) > 1 and len({file_digest(p) for p in hits}) != 1:
        raise ValueError(f'Несогласованные копии {rel}: {hits}')
    dest.parent.mkdir(parents=True, exist_ok=True)
    temp = dest.with_suffix('.copying')
    shutil.copyfile(hits[0], temp)
    temp.replace(dest)
    return dest

def require(rel):
    path = existing(rel)
    if path is None:
        raise FileNotFoundError(f'Подключите результат {rel} из запуска {RUN_ID}')
    return path

def save(rel, value):
    path = ROOT / profile_path(rel)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix('.partial')
    if isinstance(value, pd.DataFrame):
        value.to_parquet(temp, index=False)
    else:
        temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)

def load_json(rel):
    return json.loads(require(rel).read_text())

def train_indices(h, boundary, regime):
    possible = np.flatnonzero(times[:len(times)-h] + h * STEP < boundary)
    days = times[possible].normalize()
    unique = days.unique()
    early = days.isin(unique[:len(unique)//4])
    five = np.isin(times[possible].hour * 60 + times[possible].minute, TRAIN_MINUTES)
    masks = {'five_starts': five, 'mixed_starts': ~early | five, 'recent_all': ~early}
    return possible[masks[regime]]

def query_indices(boundary, days):
    if days == 0:
        return np.array([len(times)-1])
    grid = pd.date_range(boundary + pd.Timedelta(hours=10, minutes=30), periods=days, freq='D')
    result = times.get_indexer(grid)
    if (result < 0).any():
        raise ValueError(f'Нет стартов для {grid}')
    return result

def metric(y, p):
    y = np.asarray(y, dtype=np.float64)
    error = np.asarray(p, dtype=np.float64) - y
    if not len(y) or not np.isfinite(error).all():
        raise ValueError('Пустая выборка или нечисловой прогноз')
    volume = float(y.sum())
    return totals(len(y), volume, float(np.abs(error).sum()), float(error.sum()))

def totals(n, volume, absolute, signed):
    score = (absolute+abs(signed))/volume if volume else None
    return dict(rows=int(n), volume=volume, absolute=absolute, signed=signed,
                WAPE=absolute/volume if volume else None, relative_bias=signed/volume if volume else None,
                score=score, **{'WAPE + |Relative Bias|':score})

def combine(rows):
    return totals(*(sum(row[k] for row in rows) for k in ('rows','volume','absolute','signed')))

def task_path(h, stage, regime, name):
    return f'h{h}/{stage}/{regime}/{name}.parquet'

def complete(rel):
    return existing(rel + '.json') is not None and existing(rel) is not None

def result_frame(h, origins):
    frame = pd.DataFrame(dict(cutoff=times[origins].repeat(len(routes)), route_id=np.tile(routes, len(origins))))
    frame['horizon'] = h
    frame['timestamp'] = frame.cutoff + h * STEP
    if origins[-1] + h < len(times):
        frame['y'] = target[origins+h].ravel()
    return frame

def finish_fit(rel, frame, pred, train_row, boundary, fit_through, seconds, device):
    if not np.isfinite(pred).all() or np.min(pred) < 0:
        raise ValueError(f'Некорректный прогноз {rel}')
    rows = {'train':train_row}
    if 'y' in frame:
        rows['test' if '/test/' in rel else 'held_out'] = metric(frame.y, pred)
    save(rel, frame.assign(prediction=pred))
    save(rel+'.json',dict(run_id=RUN_ID, boundary=str(boundary), fit_through=str(fit_through),
                          seconds=seconds, backend=device, metrics=rows))
    display(pd.DataFrame(rows).T.assign(task=rel))
    show_eta()

def show_eta():
    if not ETA_CONTEXT:
        return
    known = ETA_CONTEXT['completed']
    oof = [v for (h,s),v in known.items() if s.startswith('oof')]
    if not oof:
        return
    current = ETA_CONTEXT['current']
    remaining = 0.
    for h,stage in ETA_CONTEXT['pending']:
        if (h,stage) in known:
            continue
        similar = [v for (other,s),v in known.items() if s==stage]
        if not similar:
            similar = [v for (other,s),v in known.items() if s[:3]==stage[:3]]
        estimate = np.mean(similar or oof)
        if (h,stage)==current:
            estimate = max(0.,estimate-(time.monotonic()-ETA_CONTEXT['start']))
        remaining += estimate
    remaining += ETA_CONTEXT['future_features']*ETA_CONTEXT['features']
    print(f'ETA части по измеренным этапам: {remaining/3600:.2f} ч')

def stage_spec(stage):
    if stage.startswith('oof'):
        return START + (int(stage[-1])-1)*7*DAY, 7
    if stage == 'test':
        return TEST_START, 14
    if stage == 'extra':
        return TEST_START + 7*DAY, 7
    if stage == 'full':
        return times[-1] + STEP, 0
    raise ValueError(stage)

def device_setup():
    global xp, cp, cupyx, BACKEND, LGB_DEVICE
    try:
        import cupy as cp
        import cupyx
        from cuml.ensemble import RandomForestRegressor
        device_name = cp.cuda.runtime.getDeviceProperties(0)['name']
    except (ImportError,RuntimeError) as error:
        raise RuntimeError('Для расчёта баз включите NVIDIA GPU и подключите CuPy и cuML, совместимые с CUDA сессии.') from error
    xp, BACKEND = cp, 'cuda'
    cp.linalg.solve(cp.eye(2),cp.ones(2))
    LGB_DEVICE = 'cpu'
    for candidate in ['cuda','gpu']:
        code = f"""import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
r = np.tile(np.arange({len(routes)}),32)
rng = np.random.default_rng(42)
data = pd.DataFrame(dict(route_id=pd.Categorical(r),volume=rng.random(len(r))))
model = LGBMRegressor(device_type='{candidate}',n_estimators=2,max_cat_to_onehot=1,verbosity=-1,n_jobs={THREADS})
model.fit(data,r%17+data.volume,categorical_feature=['route_id'])
"""
        probe = subprocess.run([sys.executable,'-c',code],capture_output=True,text=True)
        if probe.returncode == 0:
            LGB_DEVICE = candidate
            break
        print(candidate, (probe.stderr or probe.stdout)[-1200:])
    print('GPU:',device_name,'LightGBM:',LGB_DEVICE)
    from importlib.metadata import version
    save(f'part{PART}/{MODE}_devices.json',dict(gpu=str(device_name),lightgbm_device=LGB_DEVICE,
         packages={k:version(k) for k in ('numpy','pandas','xgboost','lightgbm','catboost','scikit-learn')},
         cupy=cp.__version__,cuml=__import__('cuml').__version__))

In [ ]:
import cupy as cp
import cuml
from cuml.ensemble import RandomForestRegressor
xp = cp
BACKEND = 'cuda'
INPUT = Path(INPUT_DIR)
ROOT = Path(OUTPUT_DIR)
ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = locate('train_solo_track.parquet')
TEST_PATH = locate('test_solo_track.parquet')
records = []
def record(stage, started, **details):
    cp.cuda.runtime.deviceSynchronize()
    row = dict(stage=stage, seconds=time.monotonic()-started,
               elapsed_seconds=time.monotonic()-BENCHMARK_START, **details)
    records.append(row)
    temp = ROOT / 'timings.partial.csv'
    pd.DataFrame(records).to_csv(temp, index=False)
    temp.replace(ROOT / 'timings.csv')
    print(json.dumps(row, ensure_ascii=False), flush=True)

def tick():
    cp.cuda.runtime.deviceSynchronize()
    return time.monotonic()

devices = [str(cp.cuda.runtime.getDeviceProperties(i)['name']) for i in range(cp.cuda.runtime.getDeviceCount())]
(ROOT/'environment.json').write_text(json.dumps(dict(gpus=devices, cpu_count=os.cpu_count(),
    cupy=cp.__version__, cuml=cuml.__version__, horizons=BENCHMARK_HORIZONS,
    boundary=BOUNDARY, query_days=QUERY_DAYS, params=PARAMS), indent=2))
print('GPU:', devices, 'Random Forest использует одну GPU на fit', flush=True)
load_start = tick()

In [ ]:
history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES]).sort_values(['timestamp', 'route_id'])
test = pd.read_parquet(TEST_PATH, columns=['id', 'route_id', 'timestamp'])
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError('В данных есть пропуски')
if history.duplicated(['route_id', 'timestamp']).any():
    raise ValueError('Повторяется ключ маршрута и времени')
if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
    raise ValueError('Нарушена получасовая сетка')
if not len(history) == len(times) * len(routes):
    raise ValueError('Неполная панель маршрутов')
if not (test.id.is_unique and (not test.duplicated(['route_id', 'timestamp']).any())):
    raise ValueError('Повторы идентификаторов или ключей теста')
end = times[-1]
if not (end.hour, end.minute) == (10, 30):
    raise ValueError('Последняя точка истории должна быть в 10:30')
expected = pd.MultiIndex.from_product([routes, [end + h * STEP for h in range(1, 9)]], names=['route_id', 'timestamp'])
if not expected.difference(pd.MultiIndex.from_frame(test[['route_id', 'timestamp']])).empty:
    raise ValueError('Тест не покрывает все ожидаемые маршруты и горизонты')
if not len(test) == len(expected):
    raise ValueError('Число строк теста отличается от ожидаемого')
values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
del history
if not all((np.isfinite(array).all() for array in values.values())):
    raise ValueError('В числовых рядах есть бесконечные значения')

record('load_panel', load_start)

In [ ]:
feature_start = tick()

target = values[TARGET]
if MODE != 'assemble' and ROLE != 'collect':
    target = values[TARGET]

    def past(array, positions):
        positions = np.asarray(positions, dtype=np.int64)
        valid = (positions >= 0) & (positions < len(array))
        result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
        result[~valid] = np.nan
        return result

    def average_available(matrices):
        total = np.zeros_like(matrices[0], dtype=np.float32)
        count = np.zeros_like(total)
        for matrix in matrices:
            valid = np.isfinite(matrix)
            total += np.where(valid, matrix, 0)
            count += valid
        return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

    def rolling(array, steps):
        return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
    means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
    level24 = rolling(target, 48)
    positions = np.arange(len(times))
    extra_signals = {}
    for name in STATUSES[:3]:
        extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
    extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
    extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
    for name in STATUSES[3:]:
        extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
    extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
    extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
    extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
    hub_signal = extra_signals['status_6_diff_hourly']
    extra_signals['target_vs_mean_6h'] = target - means[TARGET, 12]
    recent = means[TARGET, 4]
    usual = average_available([past(recent, positions - day * 48) for day in range(1, 8)])
    extra_signals['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)

    def offset_name(base, offset):
        if offset == 0:
            return base
        return f"{base}_offset_{('m' if offset < 0 else 'p')}{abs(offset)}"

    def add_window(common, base, array, anchor, offsets=WINDOW_OFFSETS):
        for offset in offsets:
            common[offset_name(base, offset)] = past(array, anchor + offset)

    def common_features(origins):
        common = {}
        local_target_lags = sorted({k - offset for k in (0, 1, 2, 4) for offset in WINDOW_OFFSETS if k - offset >= 0})
        for lag in local_target_lags:
            common[f'target_back_{lag}'] = past(target, origins - lag)
        causal_offsets = tuple((offset for offset in WINDOW_OFFSETS if offset <= 0))
        for name, array in values.items():
            for steps in (4, 12):
                add_window(common, f'{name}_mean_{steps}', means[name, steps], origins, causal_offsets)
            if name in STATUSES:
                common[f'{name}_latest'] = array[origins]
                for lag in range(1, 5):
                    common[f'{name}_back_{lag}'] = past(array, origins - lag)
                if name in STATUSES[:3]:
                    change = array - past(array, positions - 2)
                    add_window(common, f'{name}_change_1h', change, origins, causal_offsets)
        for name, array in extra_signals.items():
            add_window(common, name, array, origins, causal_offsets)
        for lag in (48, 96):
            add_window(common, f'target_1h_lag_{lag}', target, origins - lag)
        add_window(common, 'warehouse_total_load_lag_48', extra_signals['warehouse_total_load'], origins - 48)
        add_window(common, 'status_6_diff_hourly_lag_48', hub_signal, origins - 48)
        return common
    CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']

    for name, series in values.items():
        network = series.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None]
        for suffix, signal in [('latest', network), ('change_1h', network - past(network, positions-2))]:
            extra_signals[f'network_{name}_{suffix}'] = np.broadcast_to(signal, target.shape)


    def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
        grid = pd.DatetimeIndex(timestamps)
        route_ids = np.asarray(route_ids)
        target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
        if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
            raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
        if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
            raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
        if len(np.unique(route_ids)) != len(route_ids):
            raise ValueError('Идентификаторы маршрутов повторяются')
        steps = float(max_lookback_hours) * 2
        if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
            raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
        if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
            raise ValueError('min_pairs должен быть целым числом >=3')
        if not 0 <= min_correlation <= 1 or route_block_size < 1:
            raise ValueError('Некорректные параметры выбора корреляции')
        max_lag = int(steps)
        stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
        anchors = np.arange(max_lag + 2, stop)
        best_corr = np.full(len(route_ids), np.nan)
        best_lag = np.full(len(route_ids), np.nan)
        best_pairs = np.zeros(len(route_ids), dtype=np.int32)
        for start in range(0, len(route_ids), route_block_size):
            finish = min(start + route_block_size, len(route_ids))
            x = target_values[anchors, start:finish].astype(np.float64)
            for lag in range(2, max_lag + 1):
                z = status6_diff[anchors - lag, start:finish].astype(np.float64)
                valid = np.isfinite(x) & np.isfinite(z)
                count = valid.sum(axis=0)
                xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
                divisor = np.maximum(count, 1)
                sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
                covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
                vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
                vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
                denominator = np.sqrt(vx * vz)
                corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
                corr = np.clip(corr, -1, 1)
                previous = best_corr[start:finish]
                better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
                best_corr[start:finish][better] = corr[better]
                best_lag[start:finish][better] = lag
                best_pairs[start:finish][better] = count[better]
        accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
        return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

    def correlation_block(frame, h, lag_table, signal_array, prefix, chunk_rows=200000):
        if h not in range(1, 9) or chunk_rows < 1:
            raise ValueError('Некорректный горизонт или размер блока')
        extra_columns = [offset_name(base, offset) for offset in WINDOW_OFFSETS if offset != 0 for base in ('lagged_status6_signal', 'lagged_status6_signal_available')]
        names = CORRELATION_COLUMNS + extra_columns
        result = np.full((len(frame), len(names)), np.nan, dtype=np.float32)
        if lag_table is None:
            names = [prefix + name for name in names]
            return pd.concat([frame.drop(columns=names, errors='ignore'), pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
        by_route = lag_table.set_index('route_id').reindex(routes)
        route_index = pd.Index(routes)
        lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
        for start in range(0, len(frame), chunk_rows):
            finish = min(start + chunk_rows, len(frame))
            block = frame.iloc[start:finish]
            origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
            route = route_index.get_indexer(block.route_id)
            if (origin < 0).any() or (route < 0).any():
                raise ValueError('Строка не сопоставлена с историей')
            selected = lag[route]
            for j, name in enumerate(CORRELATION_COLUMNS[:4]):
                result[start:finish, j] = by_route[name].to_numpy(dtype=np.float32)[route]
            for offset in WINDOW_OFFSETS:
                source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64) + offset
                available = np.isfinite(selected) & (source >= 0) & (source <= origin) & (source < len(times))
                signal = signal_array[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
                available &= np.isfinite(signal)
                result[start:finish, names.index(offset_name('lagged_status6_signal', offset))] = np.where(available, signal, np.nan)
                result[start:finish, names.index(offset_name('lagged_status6_signal_available', offset))] = available.astype(np.float32)
        names = [prefix + name for name in names]
        frame = frame.drop(columns=names, errors='ignore')
        return pd.concat([frame, pd.DataFrame(result, columns=names, index=frame.index)], axis=1)
    def remove_seasonality(series):
        typical = average_available([past(series, positions - 48*d) for d in range(1,8)])
        return series - typical

    residual_target = remove_seasonality(target)
    residual_signal = remove_seasonality(hub_signal)

    def fit_lags(boundary):
        tables = []
        for y, signal, prefix in ((target,hub_signal,''),(residual_target,residual_signal,'residual_')):
            table = calculate_dynamic_shoulder(y,signal,times,routes,fit_before=boundary,max_lookback_hours=LAG_MAX_LOOKBACK_HOURS,min_pairs=LAG_MIN_PAIRS,min_correlation=LAG_MIN_CORRELATION)
            tables.append(table.rename(columns={c:prefix+c for c in table.columns if c != 'route_id'}))
        return tables[0].merge(tables[1],on='route_id',validate='one_to_one')

    def add_correlation_features(frame,h,lag_table):
        frame = correlation_block(frame,h,lag_table,hub_signal,'')
        if lag_table is None:
            return correlation_block(frame,h,None,residual_signal,'residual_')
        residual_table = lag_table[['route_id',*['residual_'+c for c in CORRELATION_COLUMNS[:4]]]].copy()
        residual_table.columns = ['route_id',*CORRELATION_COLUMNS[:4]]
        return correlation_block(frame,h,residual_table,residual_signal,'residual_')



    def horizon_dataset(h, origins):
        slot = origins + h
        dates = times[origins] + h * STEP
        common = common_features(origins)
        frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins)), **{name: matrix.ravel() for name, matrix in common.items()}})
        calendar = dict(dayofweek=dates.dayofweek, is_weekend=(dates.dayofweek >= 5).astype(np.int8))
        calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
        calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
        for name, array in calendar.items():
            frame[name] = np.repeat(array, len(routes))
        frame = add_correlation_features(frame, h, None)
        seasonal_columns = {}
        for offset in WINDOW_OFFSETS:
            for day in (1, 2, 7):
                seasonal_columns[offset_name(f'slot_days_back_{day}', offset)] = past(target, slot - day * 48 + offset).ravel()
            for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
                seasonal_columns[offset_name(name, offset)] = average_available([past(target, slot - day * 48 + offset) for day in days]).ravel()
        for base in ('slot_daily7','slot_weekly4'):
            window = np.column_stack([seasonal_columns[offset_name(base,k)] for k in WINDOW_OFFSETS])
            valid = np.isfinite(window)
            maximum = np.where(valid, window, -np.inf).max(axis=1)
            minimum = np.where(valid, window, np.inf).min(axis=1)
            peak = valid & (window == maximum[:,None])
            count = peak.sum(axis=1)
            shift = np.divide(peak @ np.asarray(WINDOW_OFFSETS, dtype=float), count, out=np.full(len(window),np.nan), where=count>0)
            seasonal_columns[base+'_peak_minutes'] = shift * 30
            seasonal_columns[base+'_range'] = np.where(count>0, maximum-minimum, np.nan)
            seasonal_columns[base+'_right_minus_left'] = window[:,5:].mean(axis=1)-window[:,:4].mean(axis=1)
        frame = pd.concat([frame, pd.DataFrame(seasonal_columns, index=frame.index)], axis=1)
        profile = np.zeros((len(origins), len(routes)))
        counts = np.zeros_like(profile)
        for day in range(1, 29):
            level = past(level24, origins - day * 48)
            adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
            profile += np.nan_to_num(adjusted)
            counts += np.isfinite(adjusted)
        frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
        if h == 1:
            lower = np.zeros_like(target[origins])
            upper = past(target, origins - 2)
            for position in (origins - 1, origins):
                total = past(target, position)
                lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
            seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
            frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
        else:
            frame['overlap'] = np.nan
        raw_target_sources = []
        for lag in range(9):
            raw_target_sources.append((f'target_back_{lag}', -lag, lag in (0, 1, 2, 4)))
        for lag in (48, 96):
            for offset in WINDOW_OFFSETS:
                raw_target_sources.append((offset_name(f'target_1h_lag_{lag}', offset), -lag + offset, offset == 0))
        for day in (1, 2, 7):
            for offset in WINDOW_OFFSETS:
                raw_target_sources.append((offset_name(f'slot_days_back_{day}', offset), h - day * 48 + offset, offset == 0))
        seen, redundant = (set(), [])
        for name, source, central in sorted(raw_target_sources, key=lambda item: not item[2]):
            if source in seen:
                redundant.append(name)
            else:
                seen.add(source)
        frame = frame.drop(columns=redundant)
        numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
        return frame.astype({name: 'float32' for name in numeric}).copy()

record('shared_features', feature_start)

In [ ]:
class HorizonData:
    def __init__(self, folder, h, stop=None):
        self.h = h
        self.stop = len(times) if stop is None else stop
        self.folder = Path(folder)
        self.corr_names = None
        self.corr = None
        self.frame = None
        self.baseline = np.empty((self.stop,len(routes)),np.float32)
        self.origins = np.arange(self.stop)
        if PROFILE=='five_starts':
            grid = times[:self.stop]
            self.origins = self.origins[np.isin(grid.hour*60+grid.minute,TRAIN_MINUTES)]
        self.slots = np.full(self.stop,-1,dtype=int)
        self.slots[self.origins] = np.arange(len(self.origins))
        for start in range(0,len(self.origins),32):
            idx = self.origins[start:start+32]
            frame = horizon_dataset(h,idx)
            if self.frame is None:
                self.cols = [c for c in frame if c not in ('cutoff','timestamp','route_id','profile28','overlap')
                             and not (c.startswith(('selected_status6','status6_lag_valid','lagged_status6','residual_')))]
                self.frame = np.memmap(self.folder/'static.bin', mode='w+', dtype='float32',
                                       shape=(len(self.origins),len(routes),len(self.cols)))
            self.frame[self.slots[idx]] = frame[self.cols].to_numpy().reshape(len(idx),len(routes),-1)
            self.baseline[idx] = frame.profile28.to_numpy().reshape(len(idx),len(routes))
        self.frame.flush()

    def prepare(self, boundary, stop):
        table = fit_lags(boundary)
        origins = self.origins[self.origins<stop]
        for start in range(0,len(origins),64):
            idx = origins[start:start+64]
            keys = result_frame(self.h,idx)[['cutoff','route_id']]
            corr = add_correlation_features(keys,self.h,table).drop(columns=['cutoff','route_id'])
            if self.corr is None:
                self.corr_names = corr.columns.tolist()
                self.corr = np.memmap(self.folder/'correlations.bin',mode='w+',dtype='float32',
                                      shape=(len(self.origins),len(routes),len(self.corr_names)))
            self.corr[self.slots[idx]] = corr.to_numpy().reshape(len(idx),len(routes),-1)

    def matrix(self, origins, route=None):
        slots = self.slots[origins]
        if (slots<0).any():
            raise ValueError('Запрошен старт вне выбранного профиля')
        if route is None:
            a,b = self.frame[slots], self.corr[slots]
        else:
            a,b = self.frame[slots,route], self.corr[slots,route]
        return np.concatenate([a,b],axis=-1).reshape(-1,len(self.cols)+len(self.corr_names))

def global_data(data, origins, label, only_finite):
    shape = (len(origins)*len(routes),len(data.cols)+len(data.corr_names))
    matrix = np.memmap(data.folder/f'{label}.bin',mode='w+',dtype='float32',shape=shape,order='F')
    count = 0
    keys = []
    for start in range(0,len(origins),32):
        chunk = origins[start:start+32]
        rows = result_frame(data.h,chunk)
        rows['profile28'] = data.baseline[chunk].ravel()
        keep = np.isfinite(rows.profile28.to_numpy())
        if not only_finite and not keep.all():
            raise ValueError('Неполный сезонный прогноз на оценке')
        batch = data.matrix(chunk)[keep]
        matrix[count:count+len(batch)] = batch
        count += len(batch)
        keys.append(rows.loc[keep])
    if count == 0:
        raise ValueError('Пустой train')
    return matrix[:count],pd.concat(keys,ignore_index=True)

def numeric_data(x, q, train_ids, query_ids, folder, scale):
    width = x.shape[1]
    bits = max(1,len(routes).bit_length())
    medians = np.empty(width)
    means = np.zeros(width)
    scales = np.ones(width)
    for col in range(width):
        values = x[:,col]
        valid = values[np.isfinite(values)]
        medians[col] = np.median(valid) if len(valid) else 0.
        if scale:
            filled = np.where(np.isfinite(values),values,medians[col])
            means[col] = np.mean(filled,dtype=np.float64)
            scales[col] = np.std(filled,dtype=np.float64) or 1.
    outputs = []
    for label,matrix,ids in [('train',x,train_ids),('query',q,query_ids)]:
        out = np.memmap(folder/f'encoded_{label}.bin',mode='w+',dtype='float32',
                        shape=(len(ids),width+bits))
        codes = pd.Index(routes).get_indexer(ids)+1
        for start in range(0,len(ids),16384):
            rows = slice(start,start+16384)
            block = matrix[rows]
            filled = np.where(np.isfinite(block),block,medians)
            out[rows,:width] = (filled-means)/scales
            for bit in range(bits):
                out[rows,width+bit] = (codes[rows] >> bit) & 1
        outputs.append(out)
    return outputs

def ridge_coefficients(x,y,alpha,backend):
    mean = np.mean(x,axis=0,dtype=np.float64)
    y_mean = np.mean(y,dtype=np.float64)
    gram = backend.zeros((x.shape[1],x.shape[1]),dtype=backend.float64)
    rhs = backend.zeros(x.shape[1],dtype=backend.float64)
    for start in range(0,len(x),16384):
        rows = slice(start,start+16384)
        a = backend.asarray(x[rows],dtype=backend.float64)-backend.asarray(mean)
        b = backend.asarray(y[rows]-y_mean,dtype=backend.float64)
        gram += a.T @ a
        rhs += a.T @ b
    gram += alpha*backend.eye(len(mean))
    coef = backend.linalg.solve(gram,rhs)
    coef = coef.get() if hasattr(coef,'get') else np.asarray(coef)
    return coef,float(y_mean - mean @ coef)

def linear_predict(x, coef, intercept):
    result = np.empty(len(x))
    for start in range(0,len(x),16384):
        result[start:start+16384] = x[start:start+16384] @ coef + intercept
    return result

def verify_global_ridge(x,y,alpha):
    rows = np.linspace(0,len(x)-1,min(1024,len(x)),dtype=int)
    a,b = np.asarray(x[rows],dtype=np.float64),y[rows]
    coef,offset = ridge_coefficients(a,b,alpha,xp)
    expected = Ridge(alpha=alpha,solver='lsqr',tol=1e-5).fit(a,b).predict(a)
    if not np.allclose(a @ coef+offset,expected,rtol=2e-4,atol=2e-4*max(1.,np.max(np.abs(b)))):
        raise ValueError('GPU Ridge расходится с sklearn')

def tree_frame(matrix, columns, ids, catboost):
    frame = pd.DataFrame(matrix,columns=columns,copy=False)
    frame.insert(0,'route_id',ids.astype(str).to_numpy() if catboost else pd.Categorical(ids,categories=routes))
    return frame

def fit_global(name, cfg, x,q,train,query, columns, folder, encoded=None):
    family = name.rsplit('_',1)[0]
    residual = name.endswith('_residual')
    baseline = train.profile28.to_numpy() if residual else 0.
    y = (train.y.to_numpy(dtype=np.float64)-baseline)/TARGET_SCALE
    params = cfg['params']
    if family in ('ridge','rf'):
        a,b = encoded if encoded is not None else numeric_data(x,q,train.route_id,query.route_id,folder,family=='ridge')
        if family == 'ridge':
            verify_global_ridge(a,y,params['alpha'])
            coef,intercept = ridge_coefficients(a,y,params['alpha'],xp)
            train_pred, pred = (linear_predict(z,coef,intercept) for z in (a,b))
        else:
            from cuml.ensemble import RandomForestRegressor
            model = RandomForestRegressor(random_state=42,n_streams=4,output_type='numpy',**params)
            model.fit(a,y.astype(np.float32))
            train_pred,pred = (np.concatenate([np.asarray(model.predict(z[i:i+32768])).reshape(-1)
                                               for i in range(0,len(z),32768)]) for z in (a,b))
        device = 'cuda'
    else:
        a,b = (tree_frame(z,columns,frame.route_id,family=='catboost') for z,frame in ((x,train),(q,query)))
        if family == 'xgb':
            from xgboost import XGBRegressor
            model = XGBRegressor(n_estimators=cfg['trees'],device='cuda',tree_method='hist',enable_categorical=True,
                                 max_cat_to_onehot=1,objective='reg:squarederror',random_state=42,n_jobs=THREADS,**params)
            model.fit(a,y)
            device = json.loads(model.get_booster().save_config())['learner']['generic_param']['device']
            if not device.startswith('cuda'):
                raise RuntimeError('XGBoost не использует GPU')
        elif family == 'catboost':
            from catboost import CatBoostRegressor
            model = CatBoostRegressor(iterations=cfg['trees'],loss_function='RMSE',task_type='GPU',random_seed=42,
                                      bootstrap_type='Bayesian',one_hot_max_size=1,allow_writing_files=False,
                                      thread_count=THREADS,verbose=False,**params)
            model.fit(a,y,cat_features=['route_id'])
            device = 'cuda'
        else:
            from lightgbm import LGBMRegressor
            model = LGBMRegressor(n_estimators=cfg['trees'],objective='regression',device_type=LGB_DEVICE,
                                  subsample_freq=1,max_cat_to_onehot=1,verbosity=-1,random_state=42,n_jobs=THREADS,**params)
            model.fit(a,y,categorical_feature=['route_id'])
            device = LGB_DEVICE
        train_pred,pred = (np.concatenate([model.predict(z.iloc[i:i+32768]) for i in range(0,len(z),32768)])
                           for z in (a,b))
    train_pred = np.maximum(0.,train_pred*TARGET_SCALE+baseline)
    pred = np.maximum(0.,pred*TARGET_SCALE+(query.profile28.to_numpy() if residual else 0.))
    return metric(train.y,train_pred),pred,device

In [ ]:
boundary = pd.Timestamp(BOUNDARY)
query = query_indices(boundary, QUERY_DAYS)
metrics = []
for h in BENCHMARK_HORIZONS:
    if str(h) not in PARAMS['rf_direct']:
        raise ValueError('Встроены параметры только h1 и h8')
    origins = train_indices(h, boundary, 'recent_all')
    possible = np.flatnonzero(times[:len(times)-h] + h*STEP < boundary)
    all_days = times[possible].normalize().unique()
    dropped = len(all_days)//4
    if len(origins) == 0 or not (times[origins] + h*STEP < boundary).all():
        raise ValueError('Неверная граница train')
    if not times[origins].normalize().isin(all_days[dropped:]).all():
        raise ValueError('В train остались ранние даты')
    print(dict(h=h, total_dates=len(all_days), excluded_dates=dropped,
               first_train=str(times[origins[0]]), last_label=str(times[origins[-1]]+h*STEP),
               potential_rows=len(origins)*len(routes)), flush=True)
    scratch = '/kaggle/temp' if Path('/kaggle/temp').is_dir() else None
    with TemporaryDirectory(dir=scratch) as folder:
        started = tick()
        data = HorizonData(folder, h, int(query[-1])+1)
        record('horizon_features', started, horizon=h)
        started = tick()
        data.prepare(boundary, int(query[-1])+1)
        record('correlation_features', started, horizon=h)
        started = tick()
        x, train = global_data(data, origins, 'train', True)
        q, valid = global_data(data, query, 'query', False)
        record('matrices', started, horizon=h, train_rows=len(train), test_rows=len(valid), features=x.shape[1])
        started = tick()
        a, b = numeric_data(x, q, train.route_id, valid.route_id, data.folder, False)
        record('imputation_encoding', started, horizon=h, features=a.shape[1], matrix_gib=a.nbytes/2**30)
        for variant in VARIANTS:
            name = 'rf_' + variant
            baseline = train.profile28.to_numpy() if variant == 'residual' else 0.
            y = ((train.y.to_numpy(dtype=np.float64)-baseline)/TARGET_SCALE).astype(np.float32)
            started = tick()
            model = RandomForestRegressor(random_state=42, n_streams=4, output_type='numpy',
                                           **PARAMS[name][str(h)]['params'])
            model.fit(a, y)
            record('fit', started, horizon=h, model=name)
            predictions = []
            for split, matrix, frame in [('train', a, train), ('test', b, valid)]:
                started = tick()
                pred = np.concatenate([np.asarray(model.predict(matrix[i:i+32768])).reshape(-1)
                                       for i in range(0,len(matrix),32768)])
                record('predict_'+split, started, horizon=h, model=name, rows=len(matrix))
                pred = np.maximum(0., pred*TARGET_SCALE + (frame.profile28.to_numpy() if variant=='residual' else 0.))
                if not np.isfinite(pred).all():
                    raise ValueError('Нечисловой прогноз')
                metrics.append(dict(horizon=h, model=name, split=split, **metric(frame.y, pred)))
                predictions.append(pred)
            valid.assign(prediction=predictions[1]).to_parquet(ROOT/f'h{h}_{name}_test.parquet', index=False)
            pd.DataFrame(metrics).to_csv(ROOT/'metrics.csv', index=False)
            display(pd.DataFrame(metrics).tail(2))
            del model, y, predictions, pred
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()
        del data, x, q, train, valid, a, b, matrix, frame
    gc.collect()
record('total', BENCHMARK_START)
display(pd.DataFrame(records))
print('Для оценки полного stacking нужны timings.csv и environment.json из', ROOT)